# Question 4.1

We transform each patient's time series data to a simple aggregated piece of text. To save valuable text and token space and run models faster, we came up with a
compact summary. We tried many different approaches but what we found works best is simply taking the last value for each patient. However, this might be due to the context window of these local LLMs being quite small so some of the data might be lost.

In [1]:
#Load the data
import numpy as np
import pandas as pd

df_train = pd.read_parquet("processed/set_a.parquet")
df_val   = pd.read_parquet("processed/set_b.parquet")
df_test  = pd.read_parquet("processed/set_c.parquet")

#note same function as in Exercise 2.1 to get the simple extracted features
DYNAMIC_VARS = sorted([
    'ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP',
    'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2',
    'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH'
])
STATIC_VARS = ['Age', 'Gender', 'Height', 'StaticWeight']
ALL_FEATURES = DYNAMIC_VARS + STATIC_VARS  # 41 features total

def extract_simple_features(df):
    """
    Per patient:
      - Forward-fill measurements over time to ignore gaps/NaNs.
      - Dynamic variables: take the last non-null value after ffill.
      - Static variables: take as-is (they are constant).
    Returns one row per patient with features + label.
    """
    df_sorted = df.sort_values(["RecordID", "hour"])
    # Forward-fill within each patient, then grab the final row
    last_row = (
    df_sorted
    .groupby("RecordID")
    .apply(lambda g: g.ffill().iloc[-1])
    .reset_index()
)

    features = last_row[["RecordID"] + ALL_FEATURES].copy()
    labels   = last_row[["RecordID", "In-hospital_death"]].copy()
    return features, labels

X_train, y_train = extract_simple_features(df_train)
X_val,   y_val   = extract_simple_features(df_val)
X_test,  y_test  = extract_simple_features(df_test)


In [2]:
# Short-name to full-name and unit mappings for all available variables
FEATURE_FULL_NAMES = {
    "RecordID": "ICU stay identifier",
    "hour": "Hours since ICU admission",
    "ALP": "Alkaline phosphatase",
    "ALT": "Alanine transaminase",
    "AST": "Aspartate transaminase",
    "Albumin": "Serum albumin",
    "BUN": "Blood urea nitrogen",
    "Bilirubin": "Total bilirubin",
    "Cholesterol": "Total cholesterol",
    "Creatinine": "Serum creatinine",
    "DiasABP": "Invasive diastolic arterial blood pressure",
    "FiO2": "Fraction of inspired oxygen",
    "GCS": "Glasgow Coma Score",
    "Glucose": "Serum glucose",
    "HCO3": "Serum bicarbonate",
    "HCT": "Hematocrit",
    "HR": "Heart rate",
    "K": "Serum potassium",
    "Lactate": "Serum lactate",
    "MAP": "Invasive mean arterial blood pressure",
    "MechVent": "Mechanical ventilation flag",
    "Mg": "Serum magnesium",
    "NIDiasABP": "Non-invasive diastolic arterial blood pressure",
    "NIMAP": "Non-invasive mean arterial blood pressure",
    "NISysABP": "Non-invasive systolic arterial blood pressure",
    "Na": "Serum sodium",
    "PaCO2": "Partial pressure of arterial CO2",
    "PaO2": "Partial pressure of arterial O2",
    "Platelets": "Platelet count",
    "RespRate": "Respiration rate",
    "SaO2": "Oxygen saturation of hemoglobin",
    "SysABP": "Invasive systolic arterial blood pressure",
    "Temp": "Body temperature",
    "TroponinI": "Troponin I",
    "TroponinT": "Troponin T",
    "Urine": "Urine output",
    "WBC": "White blood cell count",
    "Weight": "Observed weight",
    "pH": "Arterial pH",
    "Age": "Age",
    "Gender": "Gender (0=female, 1=male)",
    "Height": "Height",
    "StaticWeight": "Admission weight (static)",
    "ICUType": "ICU type (1=CCU, 2=CSRU, 3=MICU, 4=SICU)",
    "In-hospital_death": "In-hospital death outcome"
}

FEATURE_UNITS = {
    "RecordID": "none",
    "hour": "hour",
    "ALP": "IU/L",
    "ALT": "IU/L",
    "AST": "IU/L",
    "Albumin": "g/dL",
    "BUN": "mg/dL",
    "Bilirubin": "mg/dL",
    "Cholesterol": "mg/dL",
    "Creatinine": "mg/dL",
    "DiasABP": "mmHg",
    "FiO2": "fraction (0-1)",
    "GCS": "score (3-15)",
    "Glucose": "mg/dL",
    "HCO3": "mmol/L",
    "HCT": "percent",
    "HR": "bpm",
    "K": "mEq/L",
    "Lactate": "mmol/L",
    "MAP": "mmHg",
    "MechVent": "binary (0/1)",
    "Mg": "mmol/L",
    "NIDiasABP": "mmHg",
    "NIMAP": "mmHg",
    "NISysABP": "mmHg",
    "Na": "mEq/L",
    "PaCO2": "mmHg",
    "PaO2": "mmHg",
    "Platelets": "cells/nL",
    "RespRate": "bpm",
    "SaO2": "percent",
    "SysABP": "mmHg",
    "Temp": "degC",
    "TroponinI": "ug/L",
    "TroponinT": "ug/L",
    "Urine": "mL",
    "WBC": "cells/nL",
    "Weight": "kg",
    "pH": "pH units",
    "Age": "years",
    "Gender": "binary (0/1)",
    "Height": "cm",
    "StaticWeight": "kg",
    "ICUType": "categorical (1-4)",
    "In-hospital_death": "binary (0/1)"
}

The function below creates an LLM readable summary of a row of data. Note here that a row data already contains the feature for each variable. So it simply puts it in to text and adds the units to it.

In [3]:
# craete the summary for the LLM to predict whether the patient is alive or not

def create_summary(row):
    summary = []
    for feature in ALL_FEATURES:
        value = row[feature]
        if pd.isna(value):
            continue  # Skip features with missing values
        summary.append(f"last known {FEATURE_FULL_NAMES[feature]} value: {value} {FEATURE_UNITS[feature]}.")
    return "; ".join(summary)

## Predict 0 or 1 Value

In [5]:
from ollama import chat
import numpy as np
import pandas as pd

predictions = []
n_per_class = 1  # how many alive and how many deceased examples to include
max_patients = 2  # limit set for testing

# Indices for positives (1) and negatives (0)
pos_idx = y_train.index[y_train["In-hospital_death"] == 1]
neg_idx = y_train.index[y_train["In-hospital_death"] == 0]

# Build balanced few-shot set
few_shot_examples = pd.concat([
    X_train.loc[pos_idx].head(n_per_class),
    X_train.loc[neg_idx].head(n_per_class)]).reset_index(drop=True)
few_shot_answers = pd.concat([
    y_train.loc[pos_idx].head(n_per_class),
    y_train.loc[neg_idx].head(n_per_class)])["In-hospital_death"].reset_index(drop=True)

# Compose few-shot prompt
few_shot_prompt = (
    "Here are examples of ICU patients and whether they survived (0) or died (1):\n"
)


for i in range(len(few_shot_examples)):
    few_shot_prompt += f"Example {i+1}: "
    few_shot_prompt += create_summary(few_shot_examples.iloc[i])
    few_shot_prompt += f" -> {few_shot_answers[i]}\n"
few_shot_prompt += (
    "Based on the patient's last known measurements, predict in-hospital death.\n"
    "Output exactly one character: 0 if the patient is alive, 1 if deceased.\n"
    "No words, no punctuation, no explanation.\n"
)
print("Few-shot examples:")
print(few_shot_prompt)

# Query patients
for num in range(len(X_val)):
    if num >= max_patients:
        break

    prompt = (
        few_shot_prompt
        + "Now, predict the following patient (value either 0 or 1):\n"
        + create_summary(X_val.iloc[num])
        + "\nAnswer:"
    )

    response = chat(
        model="gemma3:1b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.1},
    )

    text = response.message.content.strip()
    if text not in {"0", "1"}:
        text = "1" if "1" in text else "0"
    predictions.append(int(text))
    print(f"Patient {num+1}: Predicted={text}, Actual={y_val.iloc[num]['In-hospital_death']}")

average_prediction = np.mean(predictions)
print(f"Average predicted mortality rate: {average_prediction:.2f}")

Few-shot examples:
Here are examples of ICU patients and whether they survived (0) or died (1):
Example 1: last known Alkaline phosphatase value: 47.0 IU/L.; last known Alanine transaminase value: 46.0 IU/L.; last known Aspartate transaminase value: 82.0 IU/L.; last known Serum albumin value: 1.9 g/dL.; last known Blood urea nitrogen value: 58.0 mg/dL.; last known Total bilirubin value: 0.3 mg/dL.; last known Serum creatinine value: 0.6 mg/dL.; last known Invasive diastolic arterial blood pressure value: 35.0 mmHg.; last known Fraction of inspired oxygen value: 0.4 fraction (0-1).; last known Glasgow Coma Score value: 9.0 score (3-15).; last known Serum glucose value: 116.0 mg/dL.; last known Serum bicarbonate value: 12.0 mmol/L.; last known Hematocrit value: 33.0 percent.; last known Heart rate value: 58.0 bpm.; last known Serum potassium value: 4.5 mEq/L.; last known Serum lactate value: 1.8 mmol/L.; last known Invasive mean arterial blood pressure value: 61.0 mmHg.; last known Mecha

In [6]:
#Calculate  AuROC/AuPRC for the predictions
from sklearn.metrics import roc_auc_score, average_precision_score
auroc = roc_auc_score(y_train.iloc[:max_patients]["In-hospital_death"], predictions)
auprc = average_precision_score(y_train.iloc[:max_patients]["In-hospital_death"], predictions)
print(f"AuROC: {auroc:.2f}")
print(f"AuPRC: {auprc:.2f}")

AuROC: nan
AuPRC: 0.00


c:\Users\sjaak\anaconda3\envs\ML4H\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\sjaak\anaconda3\envs\ML4H\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## Predict 0 to 10 Value

In [7]:
from ollama import chat
import numpy as np
import pandas as pd
import re

predictions = []
n_per_class = 1   # how many alive and how many deceased examples to include
max_patients = 2  # limit set for testing

# Indices for positives (1) and negatives (0)
pos_idx = y_train.index[y_train["In-hospital_death"] == 1]
neg_idx = y_train.index[y_train["In-hospital_death"] == 0]

# Build balanced few-shot set
few_shot_examples = pd.concat([
    X_train.loc[pos_idx].head(n_per_class),
    X_train.loc[neg_idx].head(n_per_class)
]).reset_index(drop=True)
few_shot_answers = pd.concat([
    y_train.loc[pos_idx].head(n_per_class),
    y_train.loc[neg_idx].head(n_per_class)
])["In-hospital_death"].reset_index(drop=True)

# Compose few-shot prompt on a 0–10 scale
few_shot_prompt = "Here are examples of ICU patients and their death risk scores (0=alive, 10=dead):\n"
for i in range(len(few_shot_examples)):
    few_shot_prompt += f"Example {i+1}: "
    few_shot_prompt += create_summary(few_shot_examples.iloc[i])
    few_shot_prompt += f" -> {10 if few_shot_answers[i] == 1 else 0}\n"
few_shot_prompt += (
    "Based on the patient's last known measurements, predict in-hospital death.\n"
    "Output a single number between 0 and 10 (decimals allowed). 0 = certain alive, 10 = certain deceased.\n"
    "No words, no punctuation, no explanation.\n"
)
print("Few-shot examples:")
print(few_shot_prompt)

# Query patients
for num in range(len(X_val)):
    if num >= max_patients:
        break

    prompt = (
        few_shot_prompt
        + "Now, predict the following patient (number between 0 and 10):\n"
        + create_summary(X_val.iloc[num])
        + "\nAnswer:"
    )

    response = chat(
        model="gemma3:1b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.1},
    )

    text = response.message.content.strip()
    # Extract first number; fall back to 5 if nothing parsable
    m = re.search(r"[0-9]*\.?[0-9]+", text)
    score = float(m.group()) if m else 5.0
    score = min(10.0, max(0.0, score))  # clamp to [0, 10]
    prob = score / 10.0                 # convert to probability
    predictions.append(prob)

    print(
        f"Patient {num+1}: Raw='{text}', Score={score:.2f}, Prob={prob:.2f}, "
        f"Actual={y_val.iloc[num]['In-hospital_death']}"
    )

average_prediction = np.mean(predictions)
print(f"Average predicted mortality rate (prob): {average_prediction:.2f}")

Few-shot examples:
Here are examples of ICU patients and their death risk scores (0=alive, 10=dead):
Example 1: last known Alkaline phosphatase value: 47.0 IU/L.; last known Alanine transaminase value: 46.0 IU/L.; last known Aspartate transaminase value: 82.0 IU/L.; last known Serum albumin value: 1.9 g/dL.; last known Blood urea nitrogen value: 58.0 mg/dL.; last known Total bilirubin value: 0.3 mg/dL.; last known Serum creatinine value: 0.6 mg/dL.; last known Invasive diastolic arterial blood pressure value: 35.0 mmHg.; last known Fraction of inspired oxygen value: 0.4 fraction (0-1).; last known Glasgow Coma Score value: 9.0 score (3-15).; last known Serum glucose value: 116.0 mg/dL.; last known Serum bicarbonate value: 12.0 mmol/L.; last known Hematocrit value: 33.0 percent.; last known Heart rate value: 58.0 bpm.; last known Serum potassium value: 4.5 mEq/L.; last known Serum lactate value: 1.8 mmol/L.; last known Invasive mean arterial blood pressure value: 61.0 mmHg.; last known 

In [9]:
#Calculate  AuROC/AuPRC for the predictions
from sklearn.metrics import roc_auc_score, average_precision_score
print(predictions)
auroc = roc_auc_score(y_train.iloc[:max_patients]["In-hospital_death"], predictions)
auprc = average_precision_score(y_train.iloc[:max_patients]["In-hospital_death"], predictions)
print(f"AuROC: {auroc:.2f}")
print(f"AuPRC: {auprc:.2f}")

[0.3, 0.0]
AuROC: nan
AuPRC: 0.00


c:\Users\sjaak\anaconda3\envs\ML4H\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\sjaak\anaconda3\envs\ML4H\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## Predict probability

In [10]:
from ollama import chat
import numpy as np
import pandas as pd
import re
from sklearn.metrics import roc_auc_score, average_precision_score

predictions = []
n_per_class = 1   # how many alive and how many deceased examples to include
max_patients = 10 # how many validation patients to score

# Indices for positives (1) and negatives (0)
pos_idx = y_train.index[y_train["In-hospital_death"] == 1]
neg_idx = y_train.index[y_train["In-hospital_death"] == 0]

# Build balanced few-shot set
few_shot_examples = pd.concat([
    X_train.loc[pos_idx].head(n_per_class),
    X_train.loc[neg_idx].head(n_per_class)
]).reset_index(drop=True)
few_shot_answers = pd.concat([
    y_train.loc[pos_idx].head(n_per_class),
    y_train.loc[neg_idx].head(n_per_class)
])["In-hospital_death"].reset_index(drop=True)

# Compose few-shot prompt that uses probabilities directly
few_shot_prompt = "Here are examples of ICU patients and their probability of in-hospital death (0 to 1):\n"
for i in range(len(few_shot_examples)):
    prob_label = 1.0 if few_shot_answers[i] == 1 else 0.0
    few_shot_prompt += f"Example {i+1}: "
    few_shot_prompt += create_summary(few_shot_examples.iloc[i])
    few_shot_prompt += f" -> {prob_label}\n"
few_shot_prompt += (
    "Based on the patient's last known measurements, predict the probability of in-hospital death.\n"
    "Output a single number between 0 and 1 (decimals allowed). No extra text.\n"
)

print("Few-shot examples:")
print(few_shot_prompt)

# Query validation patients
for num in range(len(X_val)):
    if num >= max_patients:
        break

    prompt = (
        few_shot_prompt
        + "Now, predict the following patient (probability 0–1):\n"
        + create_summary(X_val.iloc[num])
        + "\nAnswer:"
    )

    response = chat(
        model="gemma3:1b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.1},
    )

    text = response.message.content.strip()
    m = re.search(r"[0-9]*\.?[0-9]+", text)
    prob = float(m.group()) if m else 0.5
    prob = min(1.0, max(0.0, prob))  # clamp to [0,1]
    predictions.append(prob)

    print(
        f"Patient {num+1}: Raw='{text}', Prob={prob:.2f}, "
        f"Actual={y_val.iloc[num]['In-hospital_death']}"
    )

average_prediction = np.mean(predictions)
print(f"Average predicted mortality rate (prob): {average_prediction:.2f}")

# Calculate metrics on the same slice of y_val
auroc = roc_auc_score(y_val.iloc[:max_patients]["In-hospital_death"], predictions)
auprc = average_precision_score(y_val.iloc[:max_patients]["In-hospital_death"], predictions)
print(f"AuROC: {auroc:.2f}")
print(f"AuPRC: {auprc:.2f}")

Few-shot examples:
Here are examples of ICU patients and their probability of in-hospital death (0 to 1):
Example 1: last known Alkaline phosphatase value: 47.0 IU/L.; last known Alanine transaminase value: 46.0 IU/L.; last known Aspartate transaminase value: 82.0 IU/L.; last known Serum albumin value: 1.9 g/dL.; last known Blood urea nitrogen value: 58.0 mg/dL.; last known Total bilirubin value: 0.3 mg/dL.; last known Serum creatinine value: 0.6 mg/dL.; last known Invasive diastolic arterial blood pressure value: 35.0 mmHg.; last known Fraction of inspired oxygen value: 0.4 fraction (0-1).; last known Glasgow Coma Score value: 9.0 score (3-15).; last known Serum glucose value: 116.0 mg/dL.; last known Serum bicarbonate value: 12.0 mmol/L.; last known Hematocrit value: 33.0 percent.; last known Heart rate value: 58.0 bpm.; last known Serum potassium value: 4.5 mEq/L.; last known Serum lactate value: 1.8 mmol/L.; last known Invasive mean arterial blood pressure value: 61.0 mmHg.; last k

In [11]:
#Calculate  AuROC/AuPRC for the predictions
from sklearn.metrics import roc_auc_score, average_precision_score
print(predictions)
auroc = roc_auc_score(y_train.iloc[:max_patients]["In-hospital_death"], predictions)
auprc = average_precision_score(y_train.iloc[:max_patients]["In-hospital_death"], predictions)
print(f"AuROC: {auroc:.2f}")
print(f"AuPRC: {auprc:.2f}")

[0.3, 0.2, 0.6, 0.3, 0.3, 0.75, 0.3, 0.5, 0.75, 0.6]
AuROC: 0.56
AuPRC: 0.20
